# SCOPE inference from prepared features
This notebook runs the configurable SCOPE CPU prediction entry in an isolated Python 3.10 environment. It does not regenerate PSI-BLAST, HHblits or ProtT5 features. The separate Linux pipeline supports feature generation with configured external resources.

**Validation status:** Prepared-feature CPU inference was executed by the author in Google Colab on 2026-11-15 after interactive fixes. The 245-residue output differed from the archived CPU reference by at most 8e-8, with no classification changes at 0.5. This revised notebook fixes upload naming and variable consistency; a fresh uninterrupted run of this revised file remains to be checked. The underlying packaged pipeline was validated in an independent Linux environment.

Required trusted files: `scope_reproduction_candidate_20261028.tar.gz`, `best.pth`, and a ZIP containing a complete prepared feature directory. Obtain the release files from the authors. This candidate does not assume they have already been published on GitHub. Use CPU runtime; a GPU is not required.

In [ ]:
import sys, subprocess, pathlib, shutil, json, hashlib, tarfile, zipfile, datetime
ROOT = pathlib.Path('/content/scope_demo')
ROOT.mkdir(exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
UV = shutil.which('uv')
assert UV, 'uv installation failed'
subprocess.run([UV, 'python', 'install', '3.10'], check=True)
ENV = ROOT / 'venv'
if not ENV.exists():
    subprocess.run([UV, 'venv', '--python', '3.10', str(ENV)], check=True)
PY = ENV / 'bin/python'
subprocess.run([UV, 'pip', 'install', '--python', str(PY), 'torch==2.0.1+cpu', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
subprocess.run([UV, 'pip', 'install', '--python', str(PY), 'numpy==1.24.4', 'pandas==2.0.3', 'h5py==3.10.0', 'tqdm==4.66.5', 'biopython==1.83', 'transformers==4.30.2', 'sentencepiece==0.2.0'], check=True)
subprocess.run([UV, 'pip', 'check', '--python', str(PY)], check=True)


## Upload the validated source archive and checkpoint
Select the archive and `best.pth` together. Uploaded checkpoints must be trusted. Installation errors should be retained in the output; do not silently substitute model versions.

In [ ]:
from google.colab import files
uploaded = files.upload()
expected_archive_sha256 = 'f0be07217fe17966c771fee81d5dfa483af86c460155965715ab51a442662c9b'
expected_checkpoint_sha256 = 'a90c5fa95253db478c87f957657d9f48eae449aac5dc64aa1936e48a2f7cdc6f'
for name, expected_hash in [('scope_reproduction_candidate_20261028.tar.gz', expected_archive_sha256), ('best.pth', expected_checkpoint_sha256)]:
    matches = [data for data in uploaded.values() if hashlib.sha256(data).hexdigest() == expected_hash]
    assert matches, 'Missing or unexpected file: ' + name
    (ROOT / name).write_bytes(matches[0])
archive = ROOT / 'scope_reproduction_candidate_20261028.tar.gz'
with tarfile.open(archive) as tf:
    tf.extractall(ROOT, filter='data')
BUNDLE = ROOT / 'scope_reproduction_candidate'
assert (BUNDLE / 'scope_predict_features.py').is_file()
print('Checkpoint SHA256:', hashlib.sha256((ROOT / 'best.pth').read_bytes()).hexdigest())


## Upload prepared features
Upload one ZIP containing `pdb_ids.txt` and the directories `fasta`, `pssm/processed_pssm`, `hhm/processed_hhm`, `contact_map`, `Atom_feas/matched_atomfea`, `seqfea`, and `Prot5`. Preserve the filenames and HDF5 identifiers produced by the validated pipeline. A FASTA or PDB file alone is not a prepared feature bundle. No sequence search or feature generation takes place in this cell.

In [ ]:
uploaded_features = files.upload()
assert len(uploaded_features) == 1, 'Upload exactly one feature ZIP'
name, data = next(iter(uploaded_features.items()))
assert name.lower().endswith('.zip')
zpath = ROOT / 'prepared_features.zip'
zpath.write_bytes(data)
FEATURE_BASE = ROOT / ('features_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
FEATURE_BASE.mkdir()
with zipfile.ZipFile(zpath) as z:
    for entry in z.infolist():
        dest = (FEATURE_BASE / entry.filename).resolve()
        if not dest.is_relative_to(FEATURE_BASE.resolve()): raise ValueError('Unsafe ZIP path')
        if (entry.external_attr >> 16) & 0o170000 == 0o120000: raise ValueError('ZIP symlinks are not supported')
    z.extractall(FEATURE_BASE)
roots = list(FEATURE_BASE.rglob('pdb_ids.txt'))
assert len(roots) == 1, 'Feature ZIP must contain exactly one pdb_ids.txt'
FEATURES = roots[0].parent
for directory in ['fasta', 'pssm/processed_pssm', 'hhm/processed_hhm', 'contact_map', 'Atom_feas/matched_atomfea', 'seqfea', 'Prot5']:
    assert (FEATURES / directory).is_dir(), directory
print('Proteins:', roots[0].read_text().split())


## Predict and download results

In [ ]:
OUTPUT = ROOT / ('prediction_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
command = [str(PY), str(BUNDLE / 'scope_predict_features.py'), '--features', str(FEATURES), '--checkpoint', str(ROOT / 'best.pth'), '--output', str(OUTPUT), '--device', 'cpu']
result = subprocess.run(command)
if result.returncode:
    log = OUTPUT / 'run.log'
    if log.exists(): print(log.read_text()[-12000:])
    raise RuntimeError('Inference failed; see run.log')
print((OUTPUT / 'completed.json').read_text())
manifest = {'source_sha256': expected_archive_sha256, 'checkpoint_sha256': hashlib.sha256((ROOT / 'best.pth').read_bytes()).hexdigest(), 'feature_zip_sha256': hashlib.sha256(zpath.read_bytes()).hexdigest(), 'command': command, 'scope': 'Prepared-feature CPU inference only'}
(OUTPUT / 'notebook_run_manifest.json').write_text(json.dumps(manifest, indent=2))
# Include outputs and logs, not copied executable files or symlinked input trees.
result_zip = ROOT / (OUTPUT.name + '_results.zip')
with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in [* (OUTPUT / 'InferResult').glob('*'), OUTPUT / 'run.log', OUTPUT / 'completed.json', OUTPUT / 'notebook_run_manifest.json']:
        if path.is_file(): z.write(path, path.relative_to(OUTPUT))
files.download(str(result_zip))


## Full feature generation
Use the supplied Linux `environment_cpu_candidate.yml`, external asset source/version guide, and `scope_raw_pipeline.py --config ... --output ...` for raw sequence/PDB processing. Required resources include PSI-BLAST and HHblits executables and databases, the complete ProtT5 model directory, and the SCOPE checkpoint. The validated adapter requires a matching PDB chain with sequence-based residue numbering and input length 2–1000. This notebook is not evidence that those external installations or raw-input processing have been validated in Colab.